# 单轮训练快速启动 Notebook (36/1/1 Split)

**版本**: v1.0  
**日期**: 2025-01-11  
**目标**: Jupyter环境下快速跑通单轮训练流程（3 epochs）

---

## 流程概览

1. **被试划分**: 36 train + 1 val + 1 test（按被试分层）
2. **预处理**: 逐被试z-score标准化（351维特征）
3. **训练**: 4×4096全连接网络 + 软标签交叉熵
4. **评估**: Gross Acc, NLL, Brier, F1, Top-k等指标
5. **3D还原**: 1D预测无损还原到3D空间
6. **可视化**: 混淆矩阵、3D切片对比图

---

## 使用说明

1. 修改下方的 `DATA_ROOT` 指向你的数据目录
2. 运行所有单元格
3. 查看 `runs/quickstart/` 中的结果

---

## 1. 配置与导入

In [ ]:
# 配置参数
DATA_ROOT = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d"  # 数据根目录
SEED = 42
VAL_ID = None   # 可手动指定，如 "subject037"
TEST_ID = None  # 可手动指定，如 "subject038"
EPOCHS = 3      # 快跑模式，仅3个epoch
BATCH_SIZE = 256
LR = 1e-5
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 1.0  # 梯度裁剪（防止梯度爆炸）
USE_CLASS_WEIGHTS = False  # 先关闭类别权重，可选开启
CLASS_WEIGHT_ALPHA = 0.5  # 类别权重平衡系数（0=均匀权重，1=完全反比于频率）
MAX_VOX_PER_SUBJECT = None  # 可设置如50000来限制显存
SAVE_DIR = "runs/quickstart"

print("配置参数:")
print(f"  DATA_ROOT: {DATA_ROOT}")
print(f"  SEED: {SEED}")
print(f"  EPOCHS: {EPOCHS}")
print(f"  BATCH_SIZE: {BATCH_SIZE}")
print(f"  GRAD_CLIP_NORM: {GRAD_CLIP_NORM}")
print(f"  USE_CLASS_WEIGHTS: {USE_CLASS_WEIGHTS}")
print(f"  CLASS_WEIGHT_ALPHA: {CLASS_WEIGHT_ALPHA}")
print(f"  SAVE_DIR: {SAVE_DIR}")

In [ ]:
# 导入依赖
import sys
from pathlib import Path

# 确保train_runner在路径中
sys.path.insert(0, str(Path.cwd()))

from train_runner import run_single_split

print("✓ 依赖导入成功")

## 2. 验证数据目录

In [ ]:
# 检查数据目录
data_root_path = Path(DATA_ROOT)

if not data_root_path.exists():
    raise FileNotFoundError(f"数据根目录不存在: {data_root_path}")

dir_1d = data_root_path / '1d'
dir_3d = data_root_path / '3d'

if not dir_1d.exists():
    raise FileNotFoundError(f"1D数据目录不存在: {dir_1d}")
if not dir_3d.exists():
    raise FileNotFoundError(f"3D数据目录不存在: {dir_3d}")

# 列举被试
all_1d_files = sorted(dir_1d.glob('*_1d.npz'))
all_3d_files = sorted(dir_3d.glob('*_3d.npz'))

print(f"✓ 数据目录验证通过")
print(f"  1D文件数: {len(all_1d_files)}")
print(f"  3D文件数: {len(all_3d_files)}")
print(f"\n示例被试:")
for f in all_1d_files[:5]:
    subject_id = f.stem.replace('_1d', '')
    print(f"  - {subject_id}")
if len(all_1d_files) > 5:
    print(f"  ... 还有 {len(all_1d_files) - 5} 个被试")

## 3. 运行训练

**注意**: 这将启动完整的训练流程，包括：
- 数据加载与预处理
- 模型训练（3 epochs）
- 验证集与测试集评估
- 3D预测还原
- 指标计算与可视化

运行时间约 **5-15分钟**（取决于数据规模与硬件）。

In [ ]:
# 运行训练
run_single_split(
    data_root=DATA_ROOT,
    val_id=VAL_ID,
    test_id=TEST_ID,
    seed=SEED,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM,
    use_class_weights=USE_CLASS_WEIGHTS,
    class_weight_alpha=CLASS_WEIGHT_ALPHA,
    max_vox_per_subject=MAX_VOX_PER_SUBJECT,
    save_dir=SAVE_DIR
)

## 4. 查看结果

In [ ]:
# 加载运行摘要
import json

save_dir_path = Path(SAVE_DIR)
summary_path = save_dir_path / 'run_summary.json'

if summary_path.exists():
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    
    print("=" * 80)
    print("运行摘要")
    print("=" * 80)
    print(f"时间戳: {summary['timestamp']}")
    print(f"随机种子: {summary['seed']}")
    print(f"训练轮数: {summary['epochs']}")
    print(f"最优Epoch: {summary['best_epoch']}")
    print(f"\n训练集:")
    print(f"  被试数: {summary['n_train_subjects']}")
    print(f"  体素数: {summary['n_train_voxels']:,}")
    print(f"\n验证集:")
    print(f"  体素数: {summary['n_val_voxels']:,}")
    print(f"  Gross Acc (1D): {summary['val_metrics']['gross_accuracy']:.4f}")
    print(f"  Gross Acc (3D): {summary['val_3d_gross_acc']:.4f}")
    print(f"  NLL: {summary['val_metrics']['nll']:.4f}")
    print(f"  Brier: {summary['val_metrics']['brier_score']:.4f}")
    print(f"  Macro-F1: {summary['val_metrics']['macro_f1']:.4f}")
    print(f"  Top-3 Acc: {summary['val_metrics']['top3_accuracy']:.4f}")
    print(f"  Top-5 Acc: {summary['val_metrics']['top5_accuracy']:.4f}")
    print(f"\n测试集:")
    print(f"  体素数: {summary['n_test_voxels']:,}")
    print(f"  Gross Acc (1D): {summary['test_metrics']['gross_accuracy']:.4f}")
    print(f"  Gross Acc (3D): {summary['test_3d_gross_acc']:.4f}")
    print(f"  NLL: {summary['test_metrics']['nll']:.4f}")
    print(f"  Brier: {summary['test_metrics']['brier_score']:.4f}")
    print(f"  Macro-F1: {summary['test_metrics']['macro_f1']:.4f}")
    print(f"  Top-3 Acc: {summary['test_metrics']['top3_accuracy']:.4f}")
    print(f"  Top-5 Acc: {summary['test_metrics']['top5_accuracy']:.4f}")
    print("=" * 80)
else:
    print(f"运行摘要文件不存在: {summary_path}")

In [ ]:
# Display advanced metrics from summary
if summary_path.exists():
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    
    print("=" * 80)
    print("Advanced Soft-Label Metrics")
    print("=" * 80)
    
    # Validation set advanced metrics
    print("\nValidation Set:")
    print(f"  Soft ECE: {summary['val_metrics'].get('soft_ece', 'N/A'):.4f}" if isinstance(summary['val_metrics'].get('soft_ece'), (int, float)) else "  Soft ECE: N/A")
    print(f"  Classwise ECE: {summary['val_metrics'].get('classwise_ece', 'N/A'):.4f}" if isinstance(summary['val_metrics'].get('classwise_ece'), (int, float)) else "  Classwise ECE: N/A")
    print(f"  Class Mass Error: {summary['val_metrics'].get('class_mass_error', 'N/A'):.4f}" if isinstance(summary['val_metrics'].get('class_mass_error'), (int, float)) else "  Class Mass Error: N/A")
    print(f"  AURC: {summary['val_metrics'].get('aurc', 'N/A'):.4f}" if isinstance(summary['val_metrics'].get('aurc'), (int, float)) else "  AURC: N/A")
    
    # Brier decomposition
    if 'brier_decomposition' in summary['val_metrics']:
        brier_dec = summary['val_metrics']['brier_decomposition']
        print(f"  Brier Decomposition:")
        print(f"    - Reliability: {brier_dec.get('reliability', 'N/A'):.4f}" if isinstance(brier_dec.get('reliability'), (int, float)) else "    - Reliability: N/A")
        print(f"    - Resolution: {brier_dec.get('resolution', 'N/A'):.4f}" if isinstance(brier_dec.get('resolution'), (int, float)) else "    - Resolution: N/A")
        print(f"    - Uncertainty: {brier_dec.get('uncertainty', 'N/A'):.4f}" if isinstance(brier_dec.get('uncertainty'), (int, float)) else "    - Uncertainty: N/A")
    
    # Entropy stats
    if 'entropy_stats' in summary['val_metrics']:
        entropy = summary['val_metrics']['entropy_stats']
        print(f"  Entropy Statistics:")
        print(f"    - Mean: {entropy.get('mean', 'N/A'):.4f}" if isinstance(entropy.get('mean'), (int, float)) else "    - Mean: N/A")
        print(f"    - Median: {entropy.get('median', 'N/A'):.4f}" if isinstance(entropy.get('median'), (int, float)) else "    - Median: N/A")
        print(f"    - Std: {entropy.get('std', 'N/A'):.4f}" if isinstance(entropy.get('std'), (int, float)) else "    - Std: N/A")
    
    # 3D advanced metrics
    print(f"  3D Soft Dice (Macro): {summary.get('val_3d_soft_dice_macro', 'N/A'):.4f}" if isinstance(summary.get('val_3d_soft_dice_macro'), (int, float)) else "  3D Soft Dice (Macro): N/A")
    print(f"  3D NLL: {summary.get('val_3d_nll', 'N/A'):.4f}" if isinstance(summary.get('val_3d_nll'), (int, float)) else "  3D NLL: N/A")
    print(f"  3D Brier: {summary.get('val_3d_brier', 'N/A'):.4f}" if isinstance(summary.get('val_3d_brier'), (int, float)) else "  3D Brier: N/A")
    
    # Test set advanced metrics
    print("\nTest Set:")
    print(f"  Soft ECE: {summary['test_metrics'].get('soft_ece', 'N/A'):.4f}" if isinstance(summary['test_metrics'].get('soft_ece'), (int, float)) else "  Soft ECE: N/A")
    print(f"  Classwise ECE: {summary['test_metrics'].get('classwise_ece', 'N/A'):.4f}" if isinstance(summary['test_metrics'].get('classwise_ece'), (int, float)) else "  Classwise ECE: N/A")
    print(f"  Class Mass Error: {summary['test_metrics'].get('class_mass_error', 'N/A'):.4f}" if isinstance(summary['test_metrics'].get('class_mass_error'), (int, float)) else "  Class Mass Error: N/A")
    print(f"  AURC: {summary['test_metrics'].get('aurc', 'N/A'):.4f}" if isinstance(summary['test_metrics'].get('aurc'), (int, float)) else "  AURC: N/A")
    
    # Brier decomposition
    if 'brier_decomposition' in summary['test_metrics']:
        brier_dec = summary['test_metrics']['brier_decomposition']
        print(f"  Brier Decomposition:")
        print(f"    - Reliability: {brier_dec.get('reliability', 'N/A'):.4f}" if isinstance(brier_dec.get('reliability'), (int, float)) else "    - Reliability: N/A")
        print(f"    - Resolution: {brier_dec.get('resolution', 'N/A'):.4f}" if isinstance(brier_dec.get('resolution'), (int, float)) else "    - Resolution: N/A")
        print(f"    - Uncertainty: {brier_dec.get('uncertainty', 'N/A'):.4f}" if isinstance(brier_dec.get('uncertainty'), (int, float)) else "    - Uncertainty: N/A")
    
    # Entropy stats
    if 'entropy_stats' in summary['test_metrics']:
        entropy = summary['test_metrics']['entropy_stats']
        print(f"  Entropy Statistics:")
        print(f"    - Mean: {entropy.get('mean', 'N/A'):.4f}" if isinstance(entropy.get('mean'), (int, float)) else "    - Mean: N/A")
        print(f"    - Median: {entropy.get('median', 'N/A'):.4f}" if isinstance(entropy.get('median'), (int, float)) else "    - Median: N/A")
        print(f"    - Std: {entropy.get('std', 'N/A'):.4f}" if isinstance(entropy.get('std'), (int, float)) else "    - Std: N/A")
    
    # 3D advanced metrics
    print(f"  3D Soft Dice (Macro): {summary.get('test_3d_soft_dice_macro', 'N/A'):.4f}" if isinstance(summary.get('test_3d_soft_dice_macro'), (int, float)) else "  3D Soft Dice (Macro): N/A")
    print(f"  3D NLL: {summary.get('test_3d_nll', 'N/A'):.4f}" if isinstance(summary.get('test_3d_nll'), (int, float)) else "  3D NLL: N/A")
    print(f"  3D Brier: {summary.get('test_3d_brier', 'N/A'):.4f}" if isinstance(summary.get('test_3d_brier'), (int, float)) else "  3D Brier: N/A")
    
    # Temperature scaling results
    if 'optimal_temperature' in summary:
        print("\nTemperature Scaling:")
        print(f"  Optimal Temperature: {summary['optimal_temperature']:.4f}")
        print(f"  Val NLL (before → after): {summary.get('val_nll_before_temp_scaling', 'N/A'):.4f} → {summary.get('val_nll_after_temp_scaling', 'N/A'):.4f}" if isinstance(summary.get('val_nll_before_temp_scaling'), (int, float)) else "  Val NLL: N/A")
        print(f"  Val Soft ECE (before → after): {summary.get('val_soft_ece_before_temp_scaling', 'N/A'):.4f} → {summary.get('val_soft_ece_after_temp_scaling', 'N/A'):.4f}" if isinstance(summary.get('val_soft_ece_before_temp_scaling'), (int, float)) else "  Val Soft ECE: N/A")
        print(f"  Test NLL (before → after): {summary.get('test_nll_before_temp_scaling', 'N/A'):.4f} → {summary.get('test_nll_after_temp_scaling', 'N/A'):.4f}" if isinstance(summary.get('test_nll_before_temp_scaling'), (int, float)) else "  Test NLL: N/A")
        print(f"  Test Soft ECE (before → after): {summary.get('test_soft_ece_before_temp_scaling', 'N/A'):.4f} → {summary.get('test_soft_ece_after_temp_scaling', 'N/A'):.4f}" if isinstance(summary.get('test_soft_ece_before_temp_scaling'), (int, float)) else "  Test Soft ECE: N/A")
    
    print("=" * 80)
else:
    print(f"Summary file not found: {summary_path}")

## 4.1 Advanced Soft-Label Metrics

Detailed view of advanced calibration and uncertainty metrics.

## 5. 可视化混淆矩阵

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load regular confusion matrices
val_cm_path = save_dir_path / 'confusion_val.csv'
test_cm_path = save_dir_path / 'confusion_test.csv'

if val_cm_path.exists():
    val_cm = np.loadtxt(val_cm_path, delimiter=',')
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(val_cm, cmap='Blues', vmin=0, vmax=1, cbar_kws={'label': 'Normalized Count'})
    plt.title('Validation Set Confusion Matrix (Row-Normalized)', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"Confusion matrix file not found: {val_cm_path}")

if test_cm_path.exists():
    test_cm = np.loadtxt(test_cm_path, delimiter=',')
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(test_cm, cmap='Blues', vmin=0, vmax=1, cbar_kws={'label': 'Normalized Count'})
    plt.title('Test Set Confusion Matrix (Row-Normalized)', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"Confusion matrix file not found: {test_cm_path}")

In [ ]:
# Load and visualize soft confusion matrices
val_soft_cm_path = save_dir_path / 'soft_confusion_val.csv'
test_soft_cm_path = save_dir_path / 'soft_confusion_test.csv'

if val_soft_cm_path.exists():
    val_soft_cm = np.loadtxt(val_soft_cm_path, delimiter=',')
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(val_soft_cm, cmap='viridis', vmin=0, vmax=1, 
                cbar_kws={'label': 'Probability Mass'})
    plt.title('Validation Set Soft Confusion Matrix (Row-Normalized)', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"Soft confusion matrix file not found: {val_soft_cm_path}")

if test_soft_cm_path.exists():
    test_soft_cm = np.loadtxt(test_soft_cm_path, delimiter=',')
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(test_soft_cm, cmap='viridis', vmin=0, vmax=1, 
                cbar_kws={'label': 'Probability Mass'})
    plt.title('Test Set Soft Confusion Matrix (Row-Normalized)', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"Soft confusion matrix file not found: {test_soft_cm_path}")

In [ ]:
# Display advanced metrics visualizations
from IPython.display import Image, display

figs_dir = save_dir_path / 'figs'

# List of expected visualization files (correct naming from train_runner.py)
viz_files = [
    'val_reliability_diagram.png',
    'val_risk_coverage.png',
    'val_entropy_histogram.png',
    'test_reliability_diagram.png',
    'test_risk_coverage.png',
    'test_entropy_histogram.png'
]

print("Advanced Metrics Visualizations:\n")
for viz_file in viz_files:
    viz_path = figs_dir / viz_file
    if viz_path.exists():
        print(f"--- {viz_file} ---")
        display(Image(filename=str(viz_path)))
        print()
    else:
        print(f"Visualization not found: {viz_file}\n")

## 5.2 Advanced Calibration Metrics

Visualizations for soft-label evaluation: reliability diagram (ECE), risk-coverage curve (AURC), and entropy distribution.

## 5.1 Soft Confusion Matrix

Soft confusion matrix shows probability mass transfer: M[i,j] = sum of (true_prob[i] * pred_prob[j]). This is more informative for soft-label evaluation.

## 6. 查看3D切片可视化

In [ ]:
from IPython.display import Image, display

figs_dir = save_dir_path / 'figs'

if figs_dir.exists():
    # Only show 3D slice images (exclude advanced metrics visualizations)
    all_figs = sorted(figs_dir.glob('*slice*.png'))
    
    if len(all_figs) > 0:
        print(f"Found {len(all_figs)} 3D slice images:\n")
        
        for fig_path in all_figs:
            print(f"--- {fig_path.name} ---")
            display(Image(filename=str(fig_path)))
            print()
    else:
        print("No 3D slice images found")
else:
    print(f"Image directory does not exist: {figs_dir}")

## 7. 文件结构总览

In [ ]:
# 列举输出文件
import os

def print_tree(directory, prefix="", max_depth=3, current_depth=0):
    """打印目录树"""
    if current_depth >= max_depth:
        return
    
    directory = Path(directory)
    if not directory.exists():
        return
    
    contents = sorted(directory.iterdir(), key=lambda x: (x.is_file(), x.name))
    
    for i, item in enumerate(contents):
        is_last = (i == len(contents) - 1)
        current_prefix = "└── " if is_last else "├── "
        print(f"{prefix}{current_prefix}{item.name}")
        
        if item.is_dir():
            extension = "    " if is_last else "│   "
            print_tree(item, prefix + extension, max_depth, current_depth + 1)

print("\n输出文件结构:")
print(f"{save_dir_path.name}/")
print_tree(save_dir_path, max_depth=3)

## 8. 下一步

训练完成！你已经成功跑通了单轮训练流程。接下来可以：

1. **调整超参数**: 修改 `EPOCHS`, `BATCH_SIZE`, `LR` 等参数重新训练
2. **启用类别权重**: 设置 `USE_CLASS_WEIGHTS=True` 处理类别不平衡
3. **限制显存**: 设置 `MAX_VOX_PER_SUBJECT=50000` 控制内存使用
4. **指定Val/Test**: 设置 `VAL_ID` 和 `TEST_ID` 手动选择被试
5. **扩展到LOSO/K-fold**: 基于此代码扩展为交叉验证版本
6. **温度缩放校准**: 添加后处理校准步骤

---

**相关文件**:
- `train_runner.py`: 完整训练脚本（可独立运行）
- `runs/quickstart/`: 所有输出文件
- `runs/quickstart/run_summary.json`: 运行摘要（包含所有指标）

---

**版本**: v1.0  
**日期**: 2025-01-11